# Adaptive Transfer Learning for Daily Physical Activity Monitoring

This notebook reproduces the core transfer learning framework and planned ablations/extensions from the paper:
*Daily Physical Activity Monitoring: Adaptive Learning from Multi-Source Motion Sensor Data*.

The framework consists of:
1. **Domain Similarity Computation:** IPD between target and source domains.
2. **Adaptive Pre-training:** Pre-train on source domains with similarity-weighted learning rates.
3. **Fine-tuning:** Fine-tune on the target domain.

We include the original paper experiment reproduction (binary classification task), followed by ablations on paired vs. unpaired similarity, adaptive vs. fixed learning rates, noise robustness, and an extension exploring different distance metrics (using multiclass classification).


In [ ]:
!pip install pyts==0.13.0 seaborn==0.13.2
!mkdir -p DSA
!curl -o DSA/daily+and+sports+activities.zip https://archive.ics.uci.edu/static/public/256/daily+and+sports+activities.zip
!unzip -q DSA/daily+and+sports+activities.zip -d DSA


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from pyts.metrics import dtw
from pyhealth.datasets import DSADataset, get_dataloader, split_by_patient, create_sample_dataset
from pyhealth.models.adaptive_transfer import AdaptiveTransferModel
from pyhealth.tasks import DSAActivityClassification
from pyhealth.trainer import Trainer
from pyhealth.metrics import multiclass_metrics_fn

# Configuration
SEED = 598
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DSA_ROOT = "DSA/data"
TARGET_UNIT = "LL"  # Target domain: Left Leg
ALL_UNITS = ("T", "RA", "LA", "RL", "LL")
SOURCE_UNITS = [u for u in ALL_UNITS if u != TARGET_UNIT]

BATCH_SIZE = 64
EPOCHS_PRETRAIN = 2
EPOCHS_FINETUNE = 4
BASE_LR = 1e-3

plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})


## 1. Data Loading
We load the dataset for each sensor unit and create aligned train/val/test splits to maintain the paired structure across domains.


In [ ]:
base_dsa = DSADataset(root=DSA_ROOT, num_workers=1)

template_task = DSAActivityClassification(dataset_root=DSA_ROOT, selected_units=(ALL_UNITS[0],))
template_full = base_dsa.set_task(template_task, num_workers=1)
train_ref, val_ref, test_ref = split_by_patient(template_full, [0.5, 0.25, 0.25], seed=SEED)

patient_splits = {
    "train": set(train_ref.patient_to_index),
    "val": set(val_ref.patient_to_index),
    "test": set(test_ref.patient_to_index),
}

bundles = {}
for unit in ALL_UNITS:
    full = base_dsa.set_task(
        DSAActivityClassification(dataset_root=DSA_ROOT, selected_units=(unit,)),
        num_workers=1,
    )
    bundles[unit] = {
        "train": full.subset([idx for pid in patient_splits["train"] for idx in full.patient_to_index[pid]]),
        "val": full.subset([idx for pid in patient_splits["val"] for idx in full.patient_to_index[pid]]),
        "test": full.subset([idx for pid in patient_splits["test"] for idx in full.patient_to_index[pid]]),
    }


## 2. Core Framework Helper Functions
Define the DTW distance function, IPD computation, and a generic training routine.


In [ ]:
def dtw_distance_fn(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    x_np, y_np = x.detach().cpu().numpy(), y.detach().cpu().numpy()
    if x_np.ndim == 1: x_np, y_np = x_np[None, :], y_np[None, :]
    vals = [dtw(np.ravel(a), np.ravel(b)) for a, b in zip(x_np, y_np)]
    return torch.tensor(vals, dtype=x.dtype, device=x.device)

def compute_mean_ipd(src_ds, tgt_ds, distance_fn=dtw_distance_fn, shuffle_target=False):
    model = AdaptiveTransferModel(
        dataset=bundles[TARGET_UNIT]["train"], feature_key="signal", backbone="lstm",
        distance_fn=distance_fn, use_kde_smoothing=True
    ).to(DEVICE)
    model.eval()
    
    src_loader = get_dataloader(src_ds, batch_size=BATCH_SIZE, shuffle=False)
    tgt_loader = get_dataloader(tgt_ds, batch_size=BATCH_SIZE, shuffle=shuffle_target)
    
    vals = [model.compute_ipd(s, t) for s, t in zip(src_loader, tgt_loader)]
    return float(np.mean(vals))

def train_and_evaluate(source_order, ipd_scores, data_bundles, use_adaptive_lr=True):
    model = AdaptiveTransferModel(
        dataset=data_bundles[TARGET_UNIT]["train"],
        feature_key="signal",
        backbone="lstm",
        use_similarity_weighting=use_adaptive_lr
    ).to(DEVICE)
    
    def train_step(train_ds, val_ds, epochs, lr):
        (Trainer(model=model,
                device=DEVICE,
                metrics=["accuracy"], enable_logging=False)
        .train(train_dataloader=get_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True),
               val_dataloader=get_dataloader(val_ds, batch_size=BATCH_SIZE, shuffle=False),
               epochs=epochs,
               optimizer_params={"lr": lr},
               monitor="accuracy"
        ))

    for src in source_order:
        lr = model.get_adaptive_lr(BASE_LR, 1.0 / (ipd_scores[src] + 1e-8)) if use_adaptive_lr else BASE_LR
        train_step(data_bundles[src]["train"], data_bundles[src]["val"], EPOCHS_PRETRAIN, lr)
        
    train_step(data_bundles[TARGET_UNIT]["train"], data_bundles[TARGET_UNIT]["val"], EPOCHS_FINETUNE, BASE_LR)
    
    trainer = Trainer(model=model, device=DEVICE, metrics=["accuracy"], enable_logging=False)
    return trainer.evaluate(get_dataloader(data_bundles[TARGET_UNIT]["test"], batch_size=BATCH_SIZE, shuffle=False))["accuracy"]


## 3. Original Paper Experiment Reproduction (Binary Task)
**What is happening in this experiment:**
The original paper evaluates the framework on a binary classification task (one correct activity vs. rest) to calculate the Ratio of Correct Classification (RCC). We create a binary version of the dataset for this experiment (Activity 0 vs. Rest). Since the dataset is highly imbalanced, we upsample the positive samples during training and validation to match the negative samples, and downsample the negative samples during testing to ensure balanced evaluation.

We compare three settings:
1. **No Transfer:** Training only on the target domain.
2. **Direct Transfer:** Sequential pre-training on source domains without similarity weighting, followed by fine-tuning.
3. **Adaptive IPD Transfer (Proposed):** Sequential pre-training ordered by IPD with similarity-weighted learning rates, followed by fine-tuning.


In [ ]:
# Create binary datasets (Activity 0 vs Rest)
def make_binary_bundle(bundle, pos_class=0):
    bin_bundle = {}
    import random
    for split in ["train", "val", "test"]:
        samples = []
        for s in bundle[split]:
            s_new = dict(s)
            y = int(s["label"].item()) if hasattr(s["label"], "item") else int(s["label"])
            s_new["label"] = 1 if y == pos_class else 0
            samples.append(s_new)
        pos_samples = [s for s in samples if s["label"] == 1]
        neg_samples = [s for s in samples if s["label"] == 0]
        if len(pos_samples) > 0 and len(neg_samples) > 0:
            if split in ["train", "val"]:
                num_to_add = len(neg_samples) - len(pos_samples)
                if num_to_add > 0:
                    pos_samples += random.choices(pos_samples, k=num_to_add)
            elif split == "test":
                if len(neg_samples) > len(pos_samples):
                    neg_samples = random.sample(neg_samples, len(pos_samples))
        samples = pos_samples + neg_samples
        random.shuffle(samples)
        bin_bundle[split] = create_sample_dataset(samples, {"signal": "tensor"}, {"label": "binary"}, f"bin_{split}")
    return bin_bundle

binary_bundles = {unit: make_binary_bundle(bundles[unit], pos_class=0) for unit in ALL_UNITS}

# Compute Paired IPD for Adaptive Transfer (using binary validation sets)
paired_ipd_bin = {src: compute_mean_ipd(binary_bundles[src]["val"], binary_bundles[TARGET_UNIT]["val"], shuffle_target=False) for src in SOURCE_UNITS}
paired_order_bin = sorted(SOURCE_UNITS, key=paired_ipd_bin.get)

# 1. No Transfer
no_transfer_model_bin = AdaptiveTransferModel(dataset=binary_bundles[TARGET_UNIT]["train"],
                                              feature_key="signal",
                                              backbone="lstm").to(DEVICE)
(Trainer(model=no_transfer_model_bin,
        device=DEVICE,
        metrics=["accuracy"],
        enable_logging=False)
.train(train_dataloader=get_dataloader(binary_bundles[TARGET_UNIT]["train"], batch_size=BATCH_SIZE, shuffle=True),
       val_dataloader=get_dataloader(binary_bundles[TARGET_UNIT]["val"], batch_size=BATCH_SIZE, shuffle=False),
       epochs=EPOCHS_PRETRAIN * len(SOURCE_UNITS) + EPOCHS_FINETUNE,
       optimizer_params={"lr": BASE_LR},
       monitor="accuracy"
))
trainer_no_transfer_bin = Trainer(model=no_transfer_model_bin,
                                  device=DEVICE,
                                  metrics=["accuracy"],
                                  enable_logging=False)
acc_no_transfer_bin = trainer_no_transfer_bin.evaluate(get_dataloader(binary_bundles[TARGET_UNIT]["test"],
                                                                      batch_size=BATCH_SIZE,
                                                                      shuffle=False))["accuracy"]

# 2. Direct Transfer (Fixed LR, default order)
acc_direct_transfer_bin = train_and_evaluate(SOURCE_UNITS, paired_ipd_bin, binary_bundles, use_adaptive_lr=False)

# 3. Adaptive IPD Transfer (Proposed)
acc_adaptive_transfer_bin = train_and_evaluate(paired_order_bin, paired_ipd_bin, binary_bundles, use_adaptive_lr=True)


In [ ]:
# Plotting Original Paper Experiment
plt.figure(figsize=(7, 4))
sns.barplot(x=["No Transfer", "Direct Transfer", "Adaptive IPD (Proposed)"], 
            y=[acc_no_transfer_bin, acc_direct_transfer_bin, acc_adaptive_transfer_bin], 
            palette="Set2")
plt.ylabel("Test Accuracy (RCC)")
plt.title("Original Paper Experiment (Binary Task): Transfer Baselines")
plt.ylim(0, 1.0)
for i, v in enumerate([acc_no_transfer_bin, acc_direct_transfer_bin, acc_adaptive_transfer_bin]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha='center')
plt.show()


## 4. Ablation 1: Paired vs. Unpaired Similarity (H1)
*Note: For the following ablations and extensions, we use the **multiclass** classification task (all 19 activities) to provide a more challenging evaluation setting.*

**What is happening in this experiment:**
We test the hypothesis that using paired similarity (Inter-domain Pairwise Distance, IPD) is better than using unpaired similarity. IPD leverages the fact that the multi-sensor data is synchronized (i.e., collected simultaneously from different body parts during the same activity). We compare the standard paired IPD against an unpaired similarity measure, which we simulate by shuffling the target batch to break the temporal synchronization.


In [ ]:
# Compute Paired IPD (Multiclass)
paired_ipd = {src: compute_mean_ipd(bundles[src]["val"], bundles[TARGET_UNIT]["val"], shuffle_target=False) for src in SOURCE_UNITS}
paired_order = sorted(SOURCE_UNITS, key=paired_ipd.get)

# Compute Unpaired IPD
unpaired_ipd = {src: compute_mean_ipd(bundles[src]["val"], bundles[TARGET_UNIT]["val"], shuffle_target=True) for src in SOURCE_UNITS}
unpaired_order = sorted(SOURCE_UNITS, key=unpaired_ipd.get)

acc_paired = train_and_evaluate(paired_order, paired_ipd, bundles, use_adaptive_lr=True)
acc_unpaired = train_and_evaluate(unpaired_order, unpaired_ipd, bundles, use_adaptive_lr=True)


In [ ]:
# Plotting Ablation 1
plt.figure(figsize=(6, 4))
sns.barplot(x=["Paired IPD (Proposed)", "Unpaired Similarity"], y=[acc_paired, acc_unpaired], palette="viridis")
plt.ylabel("Test Accuracy")
plt.title("Ablation 1: Paired vs. Unpaired Similarity (Multiclass)")
plt.ylim(0, 1.0)
for i, v in enumerate([acc_paired, acc_unpaired]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha='center')
plt.show()


## 5. Ablation 2: Adaptive LR vs. Fixed LR (H2)
**What is happening in this experiment:**
We test the hypothesis that using adaptive learning rates (LR) based on domain similarity (IPD) is better than using a fixed learning rate for all source domains during pre-training. We compare the proposed adaptive method (where the learning rate is scaled inversely to the IPD score, so similar domains get larger learning rates) against a baseline that uses a fixed learning rate schedule across all source domains, keeping the pre-training order the same.


In [ ]:
acc_fixed_lr_ipd_order = train_and_evaluate(paired_order, paired_ipd, bundles, use_adaptive_lr=False)


In [ ]:
# Plotting Ablation 2
plt.figure(figsize=(6, 4))
sns.barplot(x=["Adaptive LR (Proposed)", "Fixed LR"], y=[acc_paired, acc_fixed_lr_ipd_order], palette="magma")
plt.ylabel("Test Accuracy")
plt.title("Ablation 2: Adaptive vs. Fixed Learning Rate (Multiclass)")
plt.ylim(0, 1.0)
for i, v in enumerate([acc_paired, acc_fixed_lr_ipd_order]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha='center')
plt.show()


## 6. Ablation 3: Robustness to Noise (H3)
**What is happening in this experiment:**
We test the hypothesis that the IPD-based adaptive transfer framework is more robust to noisy source domains. We artificially inject Gaussian noise into one of the source domains (e.g., the 'T' sensor) and evaluate how much the downstream performance degrades. The adaptive IPD method should naturally assign a lower similarity (higher IPD) and thus a lower learning rate to the noisy domain, mitigating its negative impact on the pre-trained model compared to the fixed LR baseline.


In [ ]:
def evaluate_with_noise(model, test_ds, noise_stds):
    model.eval()
    test_loader = get_dataloader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    results = []
    
    for std in noise_stds:
        ys, ps = [], []
        for data in test_loader:
            data = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in data.items()}
            if std > 0:
                data["signal"] = data["signal"] + torch.randn_like(data["signal"]) * std
            with torch.no_grad():
                out = model(**data)
            ys.append(out["y_true"].cpu().numpy())
            ps.append(out["y_prob"].cpu().numpy())
        acc = multiclass_metrics_fn(np.concatenate(ys), np.concatenate(ps), metrics=["accuracy"])["accuracy"]
        results.append(acc)
    return results

# Train a No-Transfer baseline for comparison
no_transfer_model = AdaptiveTransferModel(dataset=bundles[TARGET_UNIT]["train"], feature_key="signal", backbone="lstm").to(DEVICE)
(Trainer(model=no_transfer_model,
        device=DEVICE,
        metrics=["accuracy"],
        enable_logging=False)
.train(train_dataloader=get_dataloader(bundles[TARGET_UNIT]["train"], batch_size=BATCH_SIZE, shuffle=True),
       val_dataloader=get_dataloader(bundles[TARGET_UNIT]["val"], batch_size=BATCH_SIZE, shuffle=False),
       epochs=EPOCHS_PRETRAIN * len(SOURCE_UNITS) + EPOCHS_FINETUNE,
       optimizer_params={"lr": BASE_LR},
       monitor="accuracy"
))

# Re-train adaptive model to evaluate
adaptive_model = AdaptiveTransferModel(dataset=bundles[TARGET_UNIT]["train"], feature_key="signal", backbone="lstm", use_similarity_weighting=True).to(DEVICE)
for src in paired_order:
    (Trainer(model=adaptive_model,
            device=DEVICE,
            metrics=["accuracy"],
            enable_logging=False)
    .train(train_dataloader=get_dataloader(bundles[src]["train"], batch_size=BATCH_SIZE, shuffle=True),
           val_dataloader=get_dataloader(bundles[src]["val"], batch_size=BATCH_SIZE, shuffle=False),
           epochs=EPOCHS_PRETRAIN,
           optimizer_params={"lr": adaptive_model.get_adaptive_lr(BASE_LR, 1.0 / (paired_ipd[src] + 1e-8))},
           monitor="accuracy"
    ))
(Trainer(model=adaptive_model,
        device=DEVICE,
        metrics=["accuracy"],
        enable_logging=False)
.train(train_dataloader=get_dataloader(bundles[TARGET_UNIT]["train"], batch_size=BATCH_SIZE, shuffle=True),
       val_dataloader=get_dataloader(bundles[TARGET_UNIT]["val"], batch_size=BATCH_SIZE, shuffle=False),
       epochs=EPOCHS_FINETUNE,
       optimizer_params={"lr": BASE_LR}, monitor="accuracy"
))

noise_levels = [0.0, 0.05, 0.1, 0.2, 0.3]
acc_noise_no_transfer = evaluate_with_noise(no_transfer_model, bundles[TARGET_UNIT]["test"], noise_levels)
acc_noise_adaptive = evaluate_with_noise(adaptive_model, bundles[TARGET_UNIT]["test"], noise_levels)


In [ ]:
# Plotting Ablation 3
plt.figure(figsize=(8, 5))
plt.plot(noise_levels, acc_noise_no_transfer, marker='o', label="No Transfer")
plt.plot(noise_levels, acc_noise_adaptive, marker='s', label="Adaptive Transfer (Proposed)")
plt.xlabel("Gaussian Noise Std Dev")
plt.ylabel("Test Accuracy")
plt.title("Ablation 3: Robustness to Input Noise (Multiclass)")
plt.legend()
plt.show()


## 7. Extension: Distance Metrics Comparison
**What is happening in this experiment:**
We explore how different distance metrics affect the IPD calculation and the downstream transfer performance. The original paper uses Dynamic Time Warping (DTW), but we compare it against other common metrics like Euclidean distance, Manhattan (L1) distance, and Cosine similarity. We visualize the IPD values computed by each metric and train models using the adaptive transfer framework based on those different IPD scores.


In [ ]:
distance_metrics = {
    "DTW": dtw_distance_fn,
    "Euclidean": "euclidean",
    "Minkowski (p=3)": lambda x, y: F.pairwise_distance(x, y, p=3)
}

metric_results = {}
ipd_heatmap_data = np.zeros((len(distance_metrics), len(SOURCE_UNITS)))

for i, (name, dist_fn) in enumerate(distance_metrics.items()):
    ipd_vals = {src: compute_mean_ipd(bundles[src]["val"], bundles[TARGET_UNIT]["val"], distance_fn=dist_fn) for src in SOURCE_UNITS}
    order = sorted(SOURCE_UNITS, key=ipd_vals.get)
    acc = train_and_evaluate(order, ipd_vals, bundles, use_adaptive_lr=True)
    metric_results[name] = acc
    
    for j, src in enumerate(SOURCE_UNITS):
        ipd_heatmap_data[i, j] = ipd_vals[src]


In [ ]:
# Plotting Extension Results
# 1. Bar Chart for Accuracy Comparison
plt.figure(figsize=(7, 4))
sns.barplot(x=list(metric_results.keys()), y=list(metric_results.values()), palette="coolwarm")
plt.ylabel("Test Accuracy")
plt.title("Extension: Impact of Distance Metrics on Accuracy (Multiclass)")
plt.ylim(0, 1.0)
for i, v in enumerate(metric_results.values()):
    plt.text(i, v + 0.02, f"{v:.3f}", ha='center')
plt.show()

# 2. Heatmap for IPD Values across Metrics
plt.figure(figsize=(8, 4))
sns.heatmap(ipd_heatmap_data, annot=True, fmt=".2f", xticklabels=SOURCE_UNITS, yticklabels=list(distance_metrics.keys()), cmap="YlGnBu")
plt.xlabel("Source Domains")
plt.ylabel("Distance Metric")
plt.title("Extension: IPD Values across Different Distance Metrics")
plt.show()
